# Data Analyst (итерация 1)

# Data Analyst Report: Exploratory Data Analysis (EDA)
## Бизнес-задача
Разработка системы бинарной классификации мошеннических вакансий (Fake Job Postings) для HR-площадки. 
**Цель:** Снизить ручную модерацию и защитить пользователей от скам-постингов. 
**Метрика-приоритет:** F1-score / Recall класса 1 (мошеннические вакансии) при контроле Precision.

## Что покажет данный EDA:
1. Оценку дисбаланса классов в целевой переменной `fraudulent`.
2. Анализ корреляций числовых и закодированных категориальных признаков с таргетом.
3. Оценку распределений ключевых признаков в разрезе классов (мошеннические vs легитимные).
4. Анализ текстовых полей: длины текстов и доли пропусков (NaN) по классам.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Инициализация глобального списка для графиков
FIGS = []

# Загрузка очищенного датасета
file_path = "/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv"
DF = pd.read_csv(file_path)

print(f"Размерность датасета: {DF.shape}")
print("\nТипы данных (dtypes):")
print(DF.dtypes.value_counts())
print("\nПервые 3 строки датасета:")
print(DF.head(3))

Размерность датасета: (17880, 35)

Типы данных (dtypes):
bool       22
str         7
int64       4
float64     2
Name: count, dtype: int64

Первые 3 строки датасета:
                                       title  ... required_education_Vocational - HS Diploma
0                           Marketing Intern  ...                                      False
1  Customer Service - Cloud Video Production  ...                                      False
2    Commissioning Machinery Assistant (CMA)  ...                                      False

[3 rows x 35 columns]


## Обзор датасета (Сплит признаков)
Разделим признаки на текстовые (оставленные as_is) и числовые (включая OHE и Frequency Encoded).

In [ ]:
# Из отчета DE мы знаем, какие колонки остались текстовыми
text_cols_expected = ['title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits']
text_cols = [c for c in text_cols_expected if c in DF.columns]
target_col = 'fraudulent'

# Остальные колонки - числовые (оригинальные, OHE, Freq Encoded)
num_cols = [c for c in DF.columns if c not in text_cols and c != target_col]

print(f"Текстовые колонки ({len(text_cols)}): {text_cols}")
print(f"Числовые/Закодированные колонки ({len(num_cols)}): {num_cols}")

print("\nОписательные статистики для числовых признаков (Top 10):")
print(DF[num_cols].describe().T[['count', 'mean', 'std', 'min', 'max']].head(10))

Текстовые колонки (7): ['title', 'location', 'department', 'company_profile', 'description', 'requirements', 'benefits']
Числовые/Закодированные колонки (27): ['telecommuting', 'has_company_logo', 'has_questions', 'industry', 'function', 'employment_type_Full-time', 'employment_type_Other', 'employment_type_Part-time', 'employment_type_Temporary', 'required_experience_Director', 'required_experience_Entry level', 'required_experience_Executive', 'required_experience_Internship', 'required_experience_Mid-Senior level', 'required_experience_Not Applicable', "required_education_Bachelor's Degree", 'required_education_Certification', 'required_education_Doctorate', 'required_education_High School or equivalent', "required_education_Master's Degree", 'required_education_Professional', 'required_education_Some College Coursework Completed', 'required_education_Some High School Coursework', 'required_education_Unspecified', 'required_education_Vocational', 'required_education_Vocational - Deg

## Распределение целевой переменной (Target)
Оценим степень дисбаланса классов, так как это критично для выбора стратегии валидации и функции потерь.

In [ ]:
target_counts = DF[target_col].value_counts()
target_pct = DF[target_col].value_counts(normalize=True) * 100

print("Распределение классов 'fraudulent' (абсолютные значения):")
print(target_counts)
print("\nРаспределение классов 'fraudulent' (в %):")
print(target_pct)

imbalance_ratio = target_counts[0] / target_counts[1] if 1 in target_counts else 0
print(f"\nImbalance Ratio (Class 0 / Class 1): {imbalance_ratio:.2f}")

# Plot 1: Bar chart распределения target
fig1 = px.bar(
    x=target_counts.index.astype(str), 
    y=target_counts.values,
    labels={'x': 'Fraudulent (0 = No, 1 = Yes)', 'y': 'Количество'},
    title='Plot 1: Распределение целевой переменной (Bar Chart)', 
    text=target_counts.values,
    color=target_counts.index.astype(str),
    color_discrete_sequence=['#2ECC71', '#E74C3C']
)
FIGS.append(fig1)

# Plot 2: Pie chart долей
fig2 = px.pie(
    names=target_counts.index.astype(str), 
    values=target_counts.values,
    title='Plot 2: Доли классов целевой переменной', 
    hole=0.4,
    color=target_counts.index.astype(str),
    color_discrete_sequence=['#2ECC71', '#E74C3C']
)
FIGS.append(fig2)

Распределение классов 'fraudulent' (абсолютные значения):
fraudulent
0    17014
1      866
Name: count, dtype: int64

Распределение классов 'fraudulent' (в %):
fraudulent
0    95.1566
1     4.8434
Name: proportion, dtype: float64

Imbalance Ratio (Class 0 / Class 1): 19.65


## Числовые признаки vs Target
Проанализируем линейные связи между числовыми/закодированными признаками и целевой переменной.

In [ ]:
# Расчет корреляций с таргетом
corr_matrix = DF[num_cols + [target_col]].corr()
target_corr = corr_matrix[target_col].drop(target_col).fillna(0)
target_corr_sorted = target_corr.abs().sort_values(ascending=False)

print("Топ-10 признаков по абсолютной корреляции с таргетом:")
print(target_corr.loc[target_corr_sorted.index].head(10))

# Plot 3: Heatmap корреляций (Топ-15 признаков)
top_features = target_corr_sorted.head(15).index.tolist()
fig3 = px.imshow(
    corr_matrix.loc[top_features + [target_col], top_features + [target_col]],
    title='Plot 3: Матрица корреляций (Топ-15 признаков + Target)',
    color_continuous_scale='RdBu_r', 
    zmin=-1, zmax=1
)
FIGS.append(fig3)

# Plot 4: Bar chart абсолютных корреляций
fig4 = px.bar(
    x=target_corr_sorted.head(15).index, 
    y=target_corr_sorted.head(15).values,
    labels={'x': 'Признак', 'y': 'Абсолютная корреляция (|r|)'},
    title='Plot 4: Топ-15 признаков по модулю корреляции с Target'
)
FIGS.append(fig4)

# Plot 5 & 6: Распределение топ-2 числовых признаков по классам
top_2_num = target_corr_sorted.head(2).index.tolist()
if len(top_2_num) >= 1:
    fig5 = px.histogram(
        DF, x=top_2_num[0], color=target_col, barmode='overlay',
        title=f'Plot 5: Распределение признака "{top_2_num[0]}" по классам',
        histnorm='probability density',
        color_discrete_sequence=['#2ECC71', '#E74C3C']
    )
    FIGS.append(fig5)
    
if len(top_2_num) >= 2:
    fig6 = px.histogram(
        DF, x=top_2_num[1], color=target_col, barmode='overlay',
        title=f'Plot 6: Распределение признака "{top_2_num[1]}" по классам',
        histnorm='probability density',
        color_discrete_sequence=['#2ECC71', '#E74C3C']
    )
    FIGS.append(fig6)

Топ-10 признаков по абсолютной корреляции с таргетом:
has_company_logo                                 -0.261971
required_education_Some High School Coursework    0.125409
has_questions                                    -0.091627
required_education_High School or equivalent      0.056274
required_education_Bachelor's Degree             -0.053970
employment_type_Part-time                         0.044686
required_experience_Entry level                   0.035212
telecommuting                                     0.034523
required_education_Certification                  0.028902
employment_type_Temporary                        -0.021853
Name: fraudulent, dtype: float64


## Категориальные признаки (OHE / Binary)
Data Engineer закодировал низкокардинальные признаки через One-Hot Encoding. Найдем бинарные колонки и посмотрим на вероятность мошенничества (mean target) при наличии/отсутствии признака.

In [ ]:
# Поиск бинарных колонок (OHE)
binary_cols = [c for c in num_cols if set(DF[c].dropna().unique()).issubset({0, 1})]
print(f"Найдено бинарных колонок: {len(binary_cols)}")

if len(binary_cols) > 0:
    bin_target_rates = []
    for c in binary_cols:
        rate = DF.groupby(c)[target_col].mean().to_dict()
        bin_target_rates.append({
            'feature': c, 
            'rate_1': rate.get(1, 0), 
            'rate_0': rate.get(0, 0)
        })
        
    bin_rates_df = pd.DataFrame(bin_target_rates)
    bin_rates_df['diff'] = abs(bin_rates_df['rate_1'] - bin_rates_df['rate_0'])
    bin_rates_df = bin_rates_df.sort_values('diff', ascending=False)
    
    print("\nТоп бинарных признаков по разнице вероятности Fraud (Class 1 vs Class 0):")
    print(bin_rates_df.head(5))
    
    # Plot 7, 8, 9: Bar charts для топ-3 бинарных признаков
    top_3_bin = bin_rates_df.head(3)['feature'].tolist()
    for i, c in enumerate(top_3_bin):
        rates = DF.groupby(c)[target_col].mean().reset_index()
        rates[c] = rates[c].astype(str)
        fig = px.bar(
            rates, x=c, y=target_col,
            title=f'Plot {7+i}: Вероятность Fraud в зависимости от "{c}"',
            labels={target_col: 'Mean Target (Fraud Probability)', c: f'Значение {c}'},
            text_auto='.3f',
            color=c,
            color_discrete_sequence=['#3498DB', '#9B59B6']
        )
        FIGS.append(fig)
else:
    print("Бинарные колонки не найдены.")

Найдено бинарных колонок: 25

Топ бинарных признаков по разнице вероятности Fraud (Class 1 vs Class 0):
                                           feature    rate_1    rate_0      diff
20  required_education_Some High School Coursework  0.740741  0.047387  0.693354
1                                 has_company_logo  0.019902  0.159290  0.139388
14                required_education_Certification  0.111765  0.047826  0.063939
22                   required_education_Vocational  0.000000  0.048567  0.048567
24      required_education_Vocational - HS Diploma  0.000000  0.048458  0.048458


## Текстовые колонки
Текстовые данные оставлены as_is. Проанализируем их длину (количество слов) и долю пропусков (NaN) в разрезе целевой переменной. Это поможет DS-специалисту понять, есть ли сигнал в самом факте наличия/отсутствия текста или его объеме.

In [ ]:
print("Анализ текстовых признаков...")
text_stats = []

for c in text_cols:
    nan_rate = DF[c].isna().mean()
    
    # Считаем количество слов (пустые значения = 0 слов)
    word_counts = DF[c].fillna('').astype(str).apply(lambda x: len(x.split()) if x.strip() else 0)
    DF[f'{c}_word_count'] = word_counts
    
    mean_wc_0 = DF[DF[target_col] == 0][f'{c}_word_count'].mean()
    mean_wc_1 = DF[DF[target_col] == 1][f'{c}_word_count'].mean()
    
    text_stats.append({
        'feature': c,
        'nan_rate_overall': nan_rate,
        'mean_words_legit (0)': mean_wc_0,
        'mean_words_fraud (1)': mean_wc_1
    })

text_stats_df = pd.DataFrame(text_stats)
print("\nСтатистика по текстовым колонкам:")
print(text_stats_df.round(3))

# Plot 10: Распределение длины текста для 'description' (или первой текстовой колонки)
desc_col = 'description' if 'description' in text_cols else text_cols[0]
fig10 = px.histogram(
    DF, x=f'{desc_col}_word_count', color=target_col, barmode='overlay',
    title=f'Plot 10: Распределение количества слов в "{desc_col}" по классам',
    histnorm='probability density',
    color_discrete_sequence=['#2ECC71', '#E74C3C']
)
FIGS.append(fig10)

# Plot 11: Доля пропусков (NaN) по классам для текстовых колонок
nan_rates_by_target = []
for c in text_cols:
    nan_0 = DF[DF[target_col] == 0][c].isna().mean()
    nan_1 = DF[DF[target_col] == 1][c].isna().mean()
    nan_rates_by_target.append({'feature': c, 'target': '0 (Legit)', 'nan_rate': nan_0})
    nan_rates_by_target.append({'feature': c, 'target': '1 (Fraud)', 'nan_rate': nan_1})

nan_df = pd.DataFrame(nan_rates_by_target)
fig11 = px.bar(
    nan_df, x='feature', y='nan_rate', color='target', barmode='group',
    title='Plot 11: Доля пропусков (NaN) в текстовых колонках по классам',
    labels={'nan_rate': 'Доля пропусков', 'feature': 'Текстовый признак'},
    color_discrete_sequence=['#2ECC71', '#E74C3C']
)
FIGS.append(fig11)

Анализ текстовых признаков...

Статистика по текстовым колонкам:
           feature  nan_rate_overall  mean_words_legit (0)  mean_words_fraud (1)
0            title             0.000                 3.749                 4.016
1         location             0.019                 3.122                 3.016
2       department             0.646                 0.488                 0.641
3  company_profile             0.185                95.651                31.709
4      description             0.000               171.041               158.748
5     requirements             0.151                79.032                58.408
6         benefits             0.403                30.018                29.452


## Сводка и выводы EDA
В данном ноутбуке проведен разведочный анализ очищенного датасета. 
Конкретные бизнес-инсайты, точные значения корреляций, метрики дисбаланса и статистические различия между группами будут автоматически извлечены из stdout-ов (print) на следующем этапе пайплайна (Analyze Node).

**Структура проведенного анализа:**
1. **Дисбаланс классов:** Оценено соотношение мошеннических и легитимных вакансий.
2. **Корреляционный анализ:** Выявлены топ-признаки, имеющие наибольшую линейную связь с таргетом.
3. **Анализ категорий:** Изучено влияние наличия/отсутствия бинарных флагов (OHE) на вероятность фрода.
4. **Анализ текстов:** Проверена гипотеза о том, что мошеннические вакансии могут отличаться по объему описания (word count) или частоте незаполненных полей (NaN rate).

In [ ]:
print(f"Всего сгенерировано графиков: {len(FIGS)}")
print("EDA успешно завершен. Данные готовы для автоматической экстракции инсайтов.")

Всего сгенерировано графиков: 0
EDA успешно завершен. Данные готовы для автоматической экстракции инсайтов.
